# Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Seed
seed = 42

# Output formats
from sklearn import set_config
set_config(
    transform_output='pandas',      # fit_transform() returns pd.DataFrame instead.
    display='text',                 # fix the error when `pipeline` prints to the output cell.
)

# 1. Load

In [2]:
from sklearn.datasets import fetch_openml

X, y = fetch_openml(
    'titanic',
    version=1,
    return_X_y=True,
    as_frame=True,
)

X = X.drop(columns=['name'])    # drop unnecessary columns.

X.head()

,pclass,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


# 2. Train-Test Split

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=seed,
)

display(y_train.value_counts() / len(y_train))
display(y_test.value_counts() / len(y_test))

df = X_train.copy()     # copy for preprocess.

survived
0    0.617956
1    0.382044
Name: count, dtype: float64

survived
0    0.618321
1    0.381679
Name: count, dtype: float64

# 3. Data Types

## 1) Num / Cat

In [4]:
df_num = df.select_dtypes(include='number')
df_cat = df.select_dtypes(exclude='number')

cols_num = df_num.columns
cols_cat = df_cat.columns

print("[df_num]")
display(df_num.info())

print("[df_cat]")
display(df_cat.info())

[df_num]
<class 'pandas.DataFrame'>
Index: 1047 entries, 999 to 668
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pclass  1047 non-null   int64  
 1   age     838 non-null    float64
 2   sibsp   1047 non-null   int64  
 3   parch   1047 non-null   int64  
 4   fare    1046 non-null   float64
 5   body    92 non-null     float64
dtypes: float64(3), int64(3)
memory usage: 57.3 KB


None

[df_cat]
<class 'pandas.DataFrame'>
Index: 1047 entries, 999 to 668
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   sex        1047 non-null   category
 1   ticket     1047 non-null   str     
 2   cabin      225 non-null    str     
 3   embarked   1047 non-null   category
 4   boat       389 non-null    str     
 5   home.dest  597 non-null    str     
dtypes: category(2), str(4)
memory usage: 63.0 KB


None

## 2) str -> category

- `dtype=str` with small categories -> `astype('categorical')`

In [5]:
n_cat_boat = len(df_cat['boat'].value_counts())
print(f"n_categories of 'boat': {n_cat_boat}")
print("Convert boat: str -> category")

# Before
memory_before = df_cat['boat'].memory_usage()
print(f"- Before: {memory_before/1_000:.2f} KB")

# After
df_cat['boat'] = df_cat['boat'].astype('category')
memory_after = df_cat['boat'].memory_usage()
print(f"- After: {memory_after/1_000:.2f} KB")

n_categories of 'boat': 26
Convert boat: str -> category
- Before: 17.46 KB
- After: 9.69 KB


# 4. Null Handling

## 1) Check

In [6]:
df_num_na = df_num.isna().mean().sort_values(ascending=False)
df_cat_na = df_cat.isna().mean().sort_values(ascending=False)

display(df_num_na)
display(df_cat_na)

body      0.912130
age       0.199618
fare      0.000955
pclass    0.000000
parch     0.000000
sibsp     0.000000
dtype: float64

cabin        0.785100
boat         0.628462
home.dest    0.429799
sex          0.000000
ticket       0.000000
embarked     0.000000
dtype: float64

## 2) Drop

> Note) Do NOT blindly drop based on training data.
> - A null distribution could be different in test data.  
> - Use a basic imputation even if there are too many or no None at all.

In [7]:
# Drop cols
df_prep = df.dropna(axis=1)     # avoid in practice.

# Drop rows
df_prep = df.dropna(subset=['age'])

## 3) Imputation

### `pd.DataFrame`

In [8]:
# fillna
df_mean   = df_num.fillna(df_num.mean())
df_median = df_num.fillna(df_num.median())
df_const  = df_num.fillna(0)
df_freq   = df.fillna(df.mode().iloc[0])    # the most freq value for each column.

# interpolate
df_linear = df_num.interpolate()    # (before + after) / 2

# ffill/bfill
df_fwd    = df.ffill()          # x -> None.
df_bwd    = df.bfill()          # None <- x.

> Note) `df.interpolate()` will keep `None` as is, if both before/after values are None.

### `SimpleImputer`

#### Basic

In [9]:
from sklearn.impute import SimpleImputer

# Parameters
imp = SimpleImputer(
    strategy='mean',                # {'mean', 'median', 'most_frequent', 'constant'}.
    fill_value=None,                # value for 'constant'.
    add_indicator=False,            # if True, append 'missingindicator_{name}' columns.
    missing_values=np.nan,          # specify which values to be considered as null.
    keep_empty_features=False       # if False, drop all-None columns.
)

# Fit
df_num_prep = imp.fit_transform(df_num)

# Attributes
imp.feature_names_in_   # name of imputed features.
imp.statistics_         # fill values for each feature.
imp.n_features_in_      # number of imputed features.

6

#### Missing indicator

In [10]:
# Imputer
imp = SimpleImputer(
    strategy='mean',
    add_indicator=True,     # if True, append 'missingindicator_{name}' columns.
)

# Fit
df_num_prep = imp.fit_transform(df_num)

# Display
display(df_num_prep.head(3))

,pclass,age,sibsp,parch,fare,body,missingindicator_age,missingindicator_fare,missingindicator_body
999,3.0,29.604316,0.0,0.0,7.7500,171.576087,1.0,0.0,1.0
392,2.0,24.000000,1.0,0.0,27.7208,171.576087,0.0,0.0,1.0
628,3.0,11.000000,4.0,2.0,31.2750,171.576087,0.0,0.0,1.0


### `KNNImputer`

- Fill in with the mean of `n_neighbors` neighbors.

In [11]:
from sklearn.impute import KNNImputer

# Parameters
imp = KNNImputer(
    n_neighbors=5,
    weights='uniform',          # {'uniform', 'distance'} or callable.
    metric='nan_euclidean',     # {'nan_euclidean'} or callable.
    add_indicator=False,
    missing_values=np.nan,
    keep_empty_features=False,
)

# Methods
df_num_prep = imp.fit_transform(df_num)

# Attributes
imp.feature_names_in_   # name of imputed features.
imp.n_features_in_      # number of imputed features.

6

### `IterativeImputer`

- Use a simple model to predict the missing value from other features.

In [12]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# Parameters.
imp = IterativeImputer(
    estimator=BayesianRidge(),      # estimator object.
    sample_posterior=False,
    max_iter=10,
    tol=1e-3,
    n_nearest_features=None,

    initial_strategy='mean',        # same as SimpleImputer.
    fill_value=None,                # same as SimpleImputer.
    imputation_order='ascending',   # ascending: fewest -> most, descending: most -> fewest, roman: left -> right, arabic: right -> left, random.
    skip_complete=False,            # if True, features did not have any missing values during `fit` will be imputed with the `initial_strategy`.
    min_value=-np.inf,              # or array-like for each feature.
    max_value=np.inf,

    missing_values=np.nan,
    add_indicator=False,
    keep_empty_features=False,
    random_state=None,
    verbose=0,                      # {0, 1, 2}.
)

# Methods.
df_num_prep = imp.fit_transform(df_num)

# Attributes.
imp.initial_imputer_        # SimpleImputer, for initialization.
imp.imputation_sequence_    # (feat_idx, neighbor_feat_idx, estimator)
imp.n_iter_
imp.n_features_in_
imp.n_features_with_missing_
imp.feature_names_in_
imp.random_state_
imp.indicator_

# 5. Outlier Detection

## Summary

### Definition
- Outlier detection: rare during training.
- Novelty detection: rare after training.

### Methods
| Name                  | Class                                  | Description                | Pros                                | Cons                                  |
| --------------------- | -------------------------------------- | -------------------------- | ----------------------------------- | ------------------------------------- |
| **IQR**               | *(no dedicated sklearn class)*         | Quartile-based bounds      | Simple, fast, interpretable         | Univariate only; weak on skewed data  |
| **Elliptic Envelope** | `sklearn.covariance.EllipticEnvelope`  | Robust elliptical boundary | Multivariate; models correlations   | Assumes elliptical Gaussian-like data |
| **Isolation Forest**  | `sklearn.ensemble.IsolationForest`     | Random-split isolation     | Scalable; no scaling usually needed | Weak on local or clustered anomalies  |
| **LOF**               | `sklearn.neighbors.LocalOutlierFactor` | Local-density comparison   | Detects local anomalies             | Scale- and neighbor-sensitive         |
| **One-Class SVM**     | `sklearn.svm.OneClassSVM`              | Kernel-based boundary      | Flexible nonlinear boundary         | Slow; tuning and scaling required     |

### How to choose
- **IQR:** A simple, single-feature outlier detection.
- **Elliptic Envelope:** A multivariate, elliptical Gaussian-like data.
- **Isolation Forest:** A general-purpose method for large tabular datasets.
- **LOF:** A local anomalies in varying-density regions.
- **One-Class SVM:** A nonlinear boundaries on smaller, scaled datasets.

## 1) IQR

- IQR: Interquartile Range.
- $IQR = Q3 - Q1$
- $Lower = Q1 - 1.5 \times IQR$
- $Upper = Q3 + 1.5 \times IQR$

In [13]:
q1  = df_num.quantile(0.25)
q3  = df_num.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df_num['is_outlier'] = (
    (df_num < lower_bound) | (upper_bound < df_num)
).any(axis=1)

df_num.head()

,pclass,age,sibsp,parch,fare,body,is_outlier
999,3,NaN,0,0,7.7500,NaN,False
392,2,24.0,1,0,27.7208,NaN,False
628,3,11.0,4,2,31.2750,NaN,True
1165,3,25.0,0,0,7.2250,NaN,False
604,3,16.0,0,0,7.6500,NaN,False


> Note) 'None' is considered normal.

## 2) Elliptic Envelope

- Suppose the Gaussian distribution
- Consider a multivariate relations, compared to the univariate-only of IQR.
- Define a **center** and **covariance** of normal samples.
  - Center: mean.
  - Covariance: correlation.
- Minimum Covariance Determinant, MCD
  - Determine which subset of the size $h$ has the lowest determinant of its covariance matrix.
- Mahalanobis distance
  - A distance representing **normalness**, considering the estimated center and covariance matrix.
- Limitations
  - Requires $n_{samples} > n_{features}^2$

In [14]:
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import StandardScaler

# Sample data
df = pd.DataFrame({
    "study_hours": [
        2, 3, 4, 5, 6, 7, 8,
        3, 4, 5, 6, 7, 8,
        4, 5, 6, 7, 8,
        20, 1, 15
    ],
    "exam_score": [
        25, 32, 40, 50, 58, 68, 78,
        30, 42, 49, 61, 70, 80,
        39, 52, 60, 72, 77,
        10, 99, 95
    ],
})

features = ["study_hours", "exam_score"]
X = df[features]
X = StandardScaler().fit_transform(X)

# Model
model = EllipticEnvelope(
    assume_centered=False,
    contamination=2 / 12,   # 2 outliers out of 12 samples.
    support_fraction=None,  # a fraction of samples used to estimate the center and covariance.
    random_state=seed,
)

# Fit
df["is_outlier"] = model.fit_predict(df)            # inlier: 1, outlier: -1.
df["decision_score"] = model.decision_function(     # decision score.
    df.drop(columns=['is_outlier']),
)

df.sort_values("decision_score").head()

,study_hours,exam_score,is_outlier,decision_score
18,20,10,-1,-13168.595561
19,1,99,-1,-2964.822943
20,15,95,-1,-985.138585
0,2,25,-1,-0.736381
16,7,72,1,1.472763


## 3) Isolation Forest

1. Randomly choose a feature and split (among min ~ max).
2. Split until it is isolated or reaches the max iteration.
3. Calculate a path length for each sample and tree.
4. Average path lenghts from all trees.
5. Shorter path is more likely to be an outlier.

> Note) A `IsolationForest` simply finds the samples far from others, not modeling a meaningful relationship.

In [15]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=200,           # 200 isolation trees.
    max_samples='auto',         # 'auto': min(256, n_samples). float or int.
    max_features=1.0,           # randomly choose from the part of all features.
    contamination='auto',       # 'auto' for original thresholding rule or float in (0, 0.5].
    random_state=seed,
)

df["is_outlier"] = model.fit_predict(X)     # inlier: 1, outlier: -1.
df["decision_score"] = model.decision_function(X)

df.sort_values("decision_score").head()

,study_hours,exam_score,is_outlier,decision_score
18,20,10,-1,-0.252013
20,15,95,-1,-0.178438
19,1,99,-1,-0.163419
0,2,25,-1,-0.080357
7,3,30,1,0.018386


## 4) Local Outlier Factor (LOF)

- Is this point located in a noticeably emptier area than the points surrounding it?

1. Find the nearest $k$ neighbors.
2. Estimate the point's local density, by **reachability distance**.  
$R(x, n) = \max\ (\min d(n, \text{n's neighbor}),\ d(x, n))$, $n$ = neighbor
3. Calculate $LOF(x) = \frac{\sum density_{neighbors} / n}{density_x}$.
4. $LOF(x) > 1 \rightarrow $ outlier.

In [16]:
from sklearn.neighbors import LocalOutlierFactor

# Model
model = LocalOutlierFactor(
    n_neighbors=20,
    algorithm='auto',
    leaf_size=30,
    metric='minkowski',
    p=2,
    metric_params=None,
    contamination='auto',
    novelty=False,
)

# Predict
df['is_outlier'] = model.fit_predict(X)
df['lof_score']  = -model.negative_outlier_factor_      # negative -> positive.

display(df.sort_values('lof_score', ascending=False).head())

# Novelty
n_train = int(0.8 * len(X))
X_train, X_test = X[:n_train], X[n_train:]

model = LocalOutlierFactor(
    n_neighbors=5,
    contamination=0.1,
    novelty=True,
)

model.fit(X_train)

pred    = model.predict(X_test)
scores  = model.decision_function(X_test)

,study_hours,exam_score,is_outlier,decision_score,lof_score
4,6,58,1,0.070822,1.004035
9,5,49,1,0.071297,1.003824
3,5,50,1,0.078002,1.003589
15,6,60,1,0.086314,1.003449
5,7,68,1,0.065061,1.003242


c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


## 5) One-Class SVM

In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

# Model
model = OneClassSVM(
    kernel="rbf",       # a common kernel for OneClassSVM.
    gamma="scale",      # calculates from the n_features and variance. smaller gamma -> each point influences broadly -> smoother boundary.
    nu=0.5,             # larger nu -> allow more samples outside.
)

# Predict.
df["is_outlier"] = model.fit_predict(X)      # inlier: positive, outlier: negative.
df["decision_score"] = model.decision_function(X)

df.sort_values("decision_score").head()

,study_hours,exam_score,is_outlier,decision_score,lof_score
18,20,10,-1,-3.202957,0.979013
20,15,95,-1,-2.629617,0.998892
19,1,99,-1,-2.600808,0.979013
0,2,25,-1,-0.655300,0.998892
7,3,30,-1,-0.250750,1.001596


# 6. Feature Scaling

## 1) Standardization

- $x' = \frac{x - \mu}{\sigma}$
> Note) Use `StandardScaler(with_std=False)` for sparse data.

In [18]:
from sklearn.preprocessing import StandardScaler

# Scaler
scaler = StandardScaler(
    with_mean=True,
    with_std=True,
)

# Fit
df_num_prep = scaler.fit_transform(df_num)

# Partial fit
scaler.partial_fit(
    df_num,
    y=None,
    sample_weight=None
)

# Attributes
scaler.mean_                # mean.
scaler.scale_               # std.
scaler.var_                 # variance.
scaler.n_features_in_       # n_features.
scaler.feature_names_in_    # feature names.
scaler.n_samples_seen_      # n_samples.

# Methods
scaler.get_params()
scaler.inverse_transform(df_num_prep)   # scale back.
scaler.set_callbacks()  # callbacks.


StandardScaler()

## 2) Min-Max

- $x' = \frac{x_{max} - x}{x_{max} - x_{min}}$
- $x' = \sigma * \frac{max - min}{min}$ (if `feature_range=(min, max)` is given)
- Robust to very small std.

In [19]:
from sklearn.preprocessing import MinMaxScaler

# Scaler
scaler = MinMaxScaler(
    feature_range=(0, 1),   # specify ranges of (min, max).
    clip=False,             # clip to `feature_range`, if given. `inverse_transform` may not work.
)

# Fit
df_num_prep = scaler.fit_transform(df_num)

# Attributes
scaler.min_             # min - X.min * scale_
scaler.scale_           # (max - min) / (X.max - X.min)
scaler.data_min_        # minimum seen in the data.
scaler.data_max_        # maximum seen in the data.
scaler.data_range_      # data_max_ - data_min_

array([  2.    ,  79.6667,   8.    ,   9.    , 512.3292, 324.    ,
         1.    ])

## 3) Max-Abs

- $x' = \frac{x}{max(|x|)}$
- Preserve 0 entries, especially in sparse data.

In [20]:
from sklearn.preprocessing import MaxAbsScaler

# Scaler
scaler = MaxAbsScaler(clip=False)       # if True, clip to [-1, 1].

## 4) Robust Scaler

- $x' = \frac{x - x_{median}}{IQR}, \: IQR = q3 - q1$
- More robust to outliers.

In [21]:
from sklearn.preprocessing import RobustScaler

# Scaler.
scaler = RobustScaler(
    with_centering=True,            # will cause an err for sparse matrices!
    with_scaling=True,              # if True, divide by IQR.
    unit_variance=False,            # if True, scale to the std of quantiles (25 ~ 75).
    quantile_range=(25.0, 75.0),    # default to 1st ~ 3rd quantile.
)

## 5) Distribution Transformers

### Quantile transformer

- Original value -> Percentile rank.
```markdown
    10    → 0.00
    12    → 0.20
    14    → 0.40
    15    → 0.50
    30    → 0.70
    100   → 0.80
    1000  → 1.00
```
- Percentile rank -> Gaussian.

| Percentile | Uniform output |     Gaussian output |
| ---------: | -------------: | ------------------: |
|       10th |           0.10 | approximately −1.28 |
|       25th |           0.25 | approximately −0.67 |
|       50th |           0.50 |                   0 |
|       75th |           0.75 | approximately +0.67 |
|       90th |           0.90 | approximately +1.28 |


In [22]:
from sklearn.preprocessing import QuantileTransformer

scaler = QuantileTransformer(
    n_quantiles=1000,
    output_distribution='uniform',      # {'uniform', 'normal'}. If 'normal', maps to Gaussian distribution.
    subsample=10_000,
    random_state=None,
)

### Power transformer

- Keep the values, but smoothly bend the number line to reduce skewness.
- Types
  - `yeo-johnson`: all range.
  - `box-cox`: only positive values.

In [23]:
from sklearn.preprocessing import PowerTransformer

scaler = PowerTransformer(
    method='yeo-johnson',       # {'yeo-johnson', 'box-cox'}.
    standardize=True,
)

## 6) Normalization

- $x' = \frac{x}{||x||_2}$
- Scaling each sample to have unit norm (length).

In [24]:
from sklearn.preprocessing import Normalizer

# Scaler
scaler = Normalizer(norm='l2')    # {'l1', 'l2', 'max'}.

# 7. Encoding

## 1) Ordinal Encoder

In [25]:
from sklearn.preprocessing import OrdinalEncoder

# Encoder
enc = OrdinalEncoder(
    categories='auto',              # or array-like.
    min_frequency=0.1,              # int | float = None. Minimum frequency below to be considered infrequent.
    max_categories=None,            # n_categories that can be created per feature.
    handle_unknown='error',         # if 'use_encoded_value', fill with `unknown_value`.
    unknown_value=None,             # a value to fill in for an unknown value.
    encoded_missing_value=np.nan,   # a value to encode a missing value.
)

# Fit
df_cat_prep = enc.fit_transform(df_cat)

# Attributes
enc.categories_
enc.infrequent_categories_          # only available when either `min_frequency` or `max_categories` is not None.
enc.n_features_in_

# Methods
enc.get_feature_names_out()         # feature names.

array(['sex', 'ticket', 'cabin', 'embarked', 'boat', 'home.dest'],
      dtype=object)

## 2) One-Hot Encoder

In [26]:
from sklearn.preprocessing import OneHotEncoder

# Encoder
enc = OneHotEncoder(
    categories='auto',          # {'auto'] | array-like.
    drop=None,                  # {'first', 'if_binary'} | array-like. Drop one category per feature.
    handle_unknown='error',     # if 'infrequent_if_exist', set all to 0 for unknown categories.
    min_frequency=0.1,          # int | float = None. Minimum frequency below to be considered infrequent.
    max_categories=None,        # n_categories that can be created per feature.
    sparse_output=False,        # default=True. If True, returns a sparse matrix.
)

# Fit
df_cat_prep = enc.fit_transform(df_cat)

# Attributes
enc.categories_
enc.infrequent_categories_      # only available when either `min_frequency` or `max_categories` is not None.
enc.drop_idx_                   # dropped idx, by `drop` parameter, if any.
enc.n_features_in_

# Methods
enc.get_feature_names_out()     # feature names.

array(['sex_female', 'sex_male', 'ticket_infrequent_sklearn', 'cabin_nan',
       'cabin_infrequent_sklearn', 'embarked_C', 'embarked_S',
       'embarked_infrequent_sklearn', 'boat_nan',
       'boat_infrequent_sklearn', 'home.dest_nan',
       'home.dest_infrequent_sklearn'], dtype=object)

## 3) Target Encoder

- Encode to the mean of target variable.
  - $x_k' = \mu_{y_k}$
- Cross fitting: k-fold encoding.

> Note) Do NOT use `target_type='auto'` when y is integer. It would consider as multi-class problem where every single y value is the different class.

In [ ]:
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import KFold

# Encoder
enc = TargetEncoder(
    categories='auto',
    target_type='auto',     # {'auto', 'continuous', 'binary', 'multiclass'}.
    smooth='auto',          # {'auto'} | float. The amount of mixing with the global target mean.
    cv=KFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    ),
)

## 4) Counter Encoder

- A number of appearance for each category.

In [28]:
from category_encoders import CountEncoder

# Encoder
enc = CountEncoder(
    cols=None,
    return_df=True,
    handle_unknown='value',     # any str value.
    handle_missing='value',     # any str value.
    normalize=False,
)

# 8. Quantization

## 1) K-bins

In [29]:
from sklearn.preprocessing import KBinsDiscretizer

quantizer = KBinsDiscretizer(
    n_bins=5,               # or array-like.
    encode='onehot',        # {'onehot', 'onehot-dense', 'ordinal'}.
    strategy='quantile',    # {'uniform', 'quantile', 'kmeans'}.
    subsample='warn',       # {'warn'} | int.
    random_state=None,
)

## 2) Binarization

In [30]:
from sklearn.preprocessing import Binarizer

binarizer = Binarizer(threshold=0.0)

# 9. Target Transformer

In [31]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression

X = np.array([[0], [1], [2], [3]])
y = np.exp(2 * X.ravel())

model = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log,
    inverse_func=np.exp,
)

model.fit(X, y)

print(model.predict([[4]]))

[2980.95798704]


# 10. Pipeline

## 1) Basic

In [ ]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Data
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)

# make_pipeline: default name
pipe = make_pipeline(
    SimpleImputer(),    # default name = 'simpleimputer'.
    StandardScaler(),
)

# Pipeline: manual name
pipe = Pipeline(
    steps=[
        ('imputer', SimpleImputer()),
        ('scaler', StandardScaler()),
    ],
)

# Fit
X_prep = pipe.fit_transform(X=X)

# Get each step
pipe[:2]            # 1st and 2nd steps.
pipe['imputer']     # access by name.

# Get feature names
pipe[0].get_feature_names_out()     # 1st out names.
pipe.get_feature_names_out()        # the last names.

# Set params
pipe[0].set_params(strategy='mean')

SimpleImputer()

## 2) Caching Transformers

- Particularly useful for **parameter search**.

In [ ]:
from tempfile import mkdtemp
from shutil import rmtree

# Cache directory
cachedir = mkdtemp()    # str for directory path.

# Init with memory
pipe = Pipeline(
    steps=[
        ('imputer', SimpleImputer()),
        ('scaler', StandardScaler())
    ],
    memory=cachedir,
)

# Use
...

# Remove the cache after using it
rmtree(cachedir)

## 3) Multiple Transformations

- Multiple transformations on the same data -> fitted individually -> concatenate.

In [34]:
from sklearn.pipeline import FeatureUnion
from sklearn.decomposition import PCA
from sklearn.decomposition import KernelPCA

combined = FeatureUnion(
    [
        ('linear_pca', PCA()),
        ('kernel_pca', KernelPCA())
    ]
)

combined

FeatureUnion(transformer_list=[('linear_pca', PCA()),
                               ('kernel_pca', KernelPCA())])

## 4) Column Transformer

- To transform numeric and categorical features at once.

In [35]:
from sklearn.datasets import fetch_openml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# Data
X, y = fetch_openml(
    'titanic',
    version=1,
    return_X_y=True,
    as_frame=True,
)
X = X.drop(columns=['name'])    # drop unnecessary columns.

# Pipeline, numerical
pipe_num = make_pipeline(
    SimpleImputer(),
    StandardScaler(),
)

# Pipeline, categorical
pipe_cat = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)

# Column transformer
preprocessor = ColumnTransformer([
    ("pipe_num", pipe_num, cols_num),   # (name, pipeline, columns).
    ("pipe_cat", pipe_cat, cols_cat),
])

# Model
model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

# Fit
model.fit(X, y)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipe_num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['pclass', 'age', 'sibsp', 'parch', 'fare', 'body'], dtype='str')),
                                                 ('pipe_cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown